# Producto Unidad 1 — AquaVigia
## Dimension U1: Celia Patricia Apaza Hilasaca

Pregunta (Brief S2): ¿Con que frecuencia se registraron mediciones fuera de rango
por estacion y cuenca, y que probabilidad tiene una estacion de salir de rango
segun su historico?

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("aquavigia-u1-patricia")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 04:27:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Arquitectura Big Data seleccionada

**Caso de negocio:** Las estaciones de monitoreo de cuencas, lagos y pozos generan
mediciones dispersas entre sensores automaticos (IoT) y muestreos de laboratorio,
sin un sistema que combine ambas fuentes para anticipar episodios de deterioro de
la calidad del agua antes de que se conviertan en un riesgo sanitario o ambiental.

**Clasificacion batch/streaming:** Ambos.
- Streaming: lecturas de sensores IoT publicadas de forma continua por estacion,
  para alertas inmediatas si una medicion sale de rango.
- Batch: historico acumulado de mediciones para calcular la frecuencia de
  mediciones fuera de rango por estacion/cuenca y entrenar un modelo de
  clasificacion (esta dimension U1).

**Arquitectura seleccionada:** Lambda — regla de decision: "si el caso
necesita historico + tiempo real -> Lambda". AquaVigia necesita una batch layer
(frecuencias historicas, entrenamiento del modelo) y una speed layer (alerta
inmediata), combinadas en una serving layer comun (Grafana).

**Tecnologias propuestas:** Kafka (ingesta de sensores IoT) -> Spark Structured
Streaming (speed layer, alertas en vivo) + Spark Batch/Jupyter (batch layer, esta
dimension) -> Data Lake en Parquet (almacenamiento) -> Grafana (visualizacion).

**Supuestos y riesgos:** Supuesto: los rangos aceptables de pH/turbidez/oxigeno se
mantienen estables en el tiempo. Riesgo: drift entre la alerta en vivo (speed layer)
y la probabilidad historica calculada en batch, si el comportamiento de una
estacion cambia bruscamente (ej. contaminacion puntual).

## 2. Transformaciones distribuidas con PySpark (S2)

In [2]:
ORIGEN_DATOS = "./data"
ARTIFACTS = "./artifacts"

df_estaciones = spark.read.csv(f"{ORIGEN_DATOS}/estaciones_agua.csv", header=True, inferSchema=True)
df_mediciones = spark.read.parquet(f"{ORIGEN_DATOS}/mediciones_calidad_agua.parquet")

print("Estaciones:", df_estaciones.count())
print("Mediciones:", df_mediciones.count())
df_mediciones.printSchema()

Estaciones: 180
Mediciones: 1250000
root
 |-- medicion_id: long (nullable = true)
 |-- estacion_id: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- hora: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- ph: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- turbidez_ntu: double (nullable = true)
 |-- oxigeno_disuelto_mgl: double (nullable = true)
 |-- conductividad_us_cm: double (nullable = true)
 |-- solidos_disueltos_ppm: double (nullable = true)
 |-- nitratos_mgl: double (nullable = true)
 |-- fosfatos_mgl: double (nullable = true)
 |-- coliformes_fecales_cfu: double (nullable = true)
 |-- observaciones: string (nullable = true)



In [3]:
from pyspark.sql.functions import col

df_criticas = (
    df_mediciones
    .select("medicion_id", "estacion_id", "canal", "ph", "oxigeno_disuelto_mgl")
    .filter((col("ph") < 6.5) | (col("ph") > 8.5))
)
df_criticas

DataFrame[medicion_id: bigint, estacion_id: string, canal: string, ph: double, oxigeno_disuelto_mgl: double]

In [4]:
df_criticas.show(10)
print("Mediciones con ph fuera de rango:", df_criticas.count())

+-----------+-----------+-----------+----+--------------------+
|medicion_id|estacion_id|      canal|  ph|oxigeno_disuelto_mgl|
+-----------+-----------+-----------+----+--------------------+
|          2|   EST-0063| sensor_iot|6.29|                 7.0|
|          3|   EST-0070| sensor_iot|6.44|                8.83|
|          9|   EST-0153|laboratorio|5.96|                3.59|
|         12|   EST-0133| sensor_iot|6.23|                 7.2|
|         35|   EST-0100| sensor_iot|6.24|                7.06|
|         36|   EST-0179| sensor_iot|6.37|                7.13|
|         40|   EST-0023| sensor_iot|5.66|                5.06|
|         52|   EST-0010| sensor_iot|9.02|                5.38|
|         53|   EST-0081| sensor_iot|6.27|                8.23|
|         55|   EST-0024| sensor_iot|6.24|                4.17|
+-----------+-----------+-----------+----+--------------------+
only showing top 10 rows
Mediciones con ph fuera de rango: 168311


In [5]:
from pyspark.sql.functions import when, lit, current_date

df_mediciones = df_mediciones.withColumn(
    "fuera_de_rango",
    when(
        (col("ph") < 6.5) | (col("ph") > 8.5) |
        (col("turbidez_ntu") > 50) |
        (col("oxigeno_disuelto_mgl") < 5),
        1
    ).otherwise(0)
).withColumn(
    "fuente_dataset", lit("Proyecto Sello - AquaVigia")
).withColumn(
    "fecha_procesado", current_date()
)

df_mediciones.select("estacion_id", "ph", "turbidez_ntu", "oxigeno_disuelto_mgl", "fuera_de_rango").show(5)

+-----------+----+------------+--------------------+--------------+
|estacion_id|  ph|turbidez_ntu|oxigeno_disuelto_mgl|fuera_de_rango|
+-----------+----+------------+--------------------+--------------+
|   EST-0133| 7.8|        3.93|                6.76|             0|
|   EST-0063|6.29|       10.34|                 7.0|             1|
|   EST-0070|6.44|        3.95|                8.83|             1|
|   EST-0004|6.98|        5.42|                5.85|             0|
|   EST-0127|7.51|       17.04|                7.04|             0|
+-----------+----+------------+--------------------+--------------+
only showing top 5 rows


In [6]:
from pyspark.sql.functions import avg, count as spark_count, sum as spark_sum

df_frecuencia_estacion = df_mediciones.groupBy("estacion_id").agg(
    spark_count("*").alias("total_mediciones"),
    spark_sum("fuera_de_rango").alias("mediciones_fuera_rango"),
)
df_frecuencia_estacion = df_frecuencia_estacion.withColumn(
    "pct_fuera_rango", col("mediciones_fuera_rango") / col("total_mediciones") * 100
)
df_frecuencia_estacion.orderBy(col("pct_fuera_rango").desc()).show(10)

[Stage 16:================================================>         (5 + 1) / 6]

+-----------+----------------+----------------------+------------------+
|estacion_id|total_mediciones|mediciones_fuera_rango|   pct_fuera_rango|
+-----------+----------------+----------------------+------------------+
|   EST-0133|            6889|                  1649|23.936710698214544|
|   EST-0136|            6886|                  1647|23.918094684867846|
|   EST-0040|            7091|                  1685|23.762515865181218|
|   EST-0098|            6992|                  1656|23.684210526315788|
|   EST-0070|            6938|                  1640|23.637936004612282|
|   EST-0047|            7001|                  1654|23.625196400514213|
|   EST-0004|            7014|                  1649| 23.51012261191902|
|   EST-0145|            7015|                  1648|23.492516037063435|
|   EST-0101|            6934|                  1625|23.435246610902798|
|   EST-0169|            6876|                  1605|23.342059336823734|
+-----------+----------------+---------------------

In [7]:
# Conteo por canal
df_mediciones.groupBy("canal", "fuera_de_rango").agg(spark_count("*").alias("cantidad")).show()

+-----------+--------------+--------+
|      canal|fuera_de_rango|cantidad|
+-----------+--------------+--------+
|laboratorio|             0|  116107|
|laboratorio|             1|   34045|
| sensor_iot|             0|  851849|
| sensor_iot|             1|  247999|
+-----------+--------------+--------+



In [8]:
from pyspark.sql.window import Window

# Funcion ventana: pct fuera de rango por estacion, SIN colapsar filas
ventana_estacion = Window.partitionBy("estacion_id")
df_con_ventana = df_mediciones.withColumn(
    "promedio_fuera_rango_estacion", avg("fuera_de_rango").over(ventana_estacion)
)
df_con_ventana.filter(col("estacion_id") == "EST-0003").select(
    "estacion_id", "fuera_de_rango", "promedio_fuera_rango_estacion"
).show(5)

+-----------+--------------+-----------------------------+
|estacion_id|fuera_de_rango|promedio_fuera_rango_estacion|
+-----------+--------------+-----------------------------+
|   EST-0003|             0|          0.22877426832802086|
|   EST-0003|             1|          0.22877426832802086|
|   EST-0003|             0|          0.22877426832802086|
|   EST-0003|             0|          0.22877426832802086|
|   EST-0003|             0|          0.22877426832802086|
+-----------+--------------+-----------------------------+
only showing top 5 rows


In [9]:
df_criticas.explain(True)

== Parsed Logical Plan ==
'Filter 'or('`<`('ph, 6.5), '`>`('ph, 8.5))
+- Project [medicion_id#25L, estacion_id#26, canal#29, ph#30, oxigeno_disuelto_mgl#33]
   +- Relation [medicion_id#25L,estacion_id#26,fecha#27,hora#28,canal#29,ph#30,temperatura_c#31,turbidez_ntu#32,oxigeno_disuelto_mgl#33,conductividad_us_cm#34,solidos_disueltos_ppm#35,nitratos_mgl#36,fosfatos_mgl#37,coliformes_fecales_cfu#38,observaciones#39] parquet

== Analyzed Logical Plan ==
medicion_id: bigint, estacion_id: string, canal: string, ph: double, oxigeno_disuelto_mgl: double
Filter ((ph#30 < 6.5) OR (ph#30 > 8.5))
+- Project [medicion_id#25L, estacion_id#26, canal#29, ph#30, oxigeno_disuelto_mgl#33]
   +- Relation [medicion_id#25L,estacion_id#26,fecha#27,hora#28,canal#29,ph#30,temperatura_c#31,turbidez_ntu#32,oxigeno_disuelto_mgl#33,conductividad_us_cm#34,solidos_disueltos_ppm#35,nitratos_mgl#36,fosfatos_mgl#37,coliformes_fecales_cfu#38,observaciones#39] parquet

== Optimized Logical Plan ==
Project [medicion_id#25

In [10]:
import re
from operator import add

rdd = df_mediciones.select("observaciones").rdd.map(lambda x: x.observaciones)
rdd = rdd.filter(lambda texto: texto is not None)

palabras = rdd.flatMap(
    lambda linea: re.sub(r"[^\wáéíóúñüÁÉÍÓÚÑÜ]", " ", linea.lower()).split()
)
pares = palabras.filter(lambda p: p != "").map(lambda palabra: (palabra, 1))
conteo = pares.reduceByKey(add)

conteo.takeOrdered(10, key=lambda x: -x[1])

[('sin', 117498),
 ('de', 63027),
 ('en', 40092),
 ('la', 29136),
 ('muestreo', 29084),
 ('olor', 29084),
 ('sedimentos', 28923),
 ('agua', 28814),
 ('algas', 28783),
 ('presencia', 28783)]

**Hallazgo (evaluacion perezosa):** `df_criticas` sola no ejecuta nada -- Spark
solo muestra `DataFrame[...]`. Recien con `.show()`/`.count()` se dispara la
ejecucion real (S2, 2.5).

**Reflexion:** el `explain(True)` confirma predicate pushdown: el filtro por rango
de `ph` baja hasta el propio lector del Parquet, y `ReadSchema` solo trae las
columnas usadas.

## 3. Calidad de datos y particionamiento analitico (S3)

In [11]:
from pyspark.sql.functions import col, count as spark_count, when

columnas_requeridas = {
    "medicion_id", "estacion_id", "fecha", "hora", "canal",
    "ph", "temperatura_c", "turbidez_ntu", "oxigeno_disuelto_mgl",
    "conductividad_us_cm", "solidos_disueltos_ppm",
}
faltantes = columnas_requeridas - set(df_mediciones.columns)
if faltantes:
    raise ValueError(f"Faltan columnas obligatorias: {sorted(faltantes)}")
print("Esquema validado: columnas obligatorias presentes.")

Esquema validado: columnas obligatorias presentes.


In [12]:
total_filas = df_mediciones.count()
nulos = df_mediciones.select([
    spark_count(when(col(c).isNull(), c)).alias(c) for c in
    ["ph","temperatura_c","turbidez_ntu","oxigeno_disuelto_mgl","conductividad_us_cm",
     "solidos_disueltos_ppm","nitratos_mgl","fosfatos_mgl","coliformes_fecales_cfu"]
]).collect()[0].asDict()
for columna, cantidad in nulos.items():
    print(f"{columna}: {cantidad} nulos ({cantidad/total_filas*100:.1f}%)")

ph: 0 nulos (0.0%)
temperatura_c: 0 nulos (0.0%)
turbidez_ntu: 0 nulos (0.0%)
oxigeno_disuelto_mgl: 0 nulos (0.0%)
conductividad_us_cm: 0 nulos (0.0%)
solidos_disueltos_ppm: 0 nulos (0.0%)
nitratos_mgl: 1099848 nulos (88.0%)
fosfatos_mgl: 1099848 nulos (88.0%)
coliformes_fecales_cfu: 1099848 nulos (88.0%)


**Filtrado — Tecnica 1 (SQL como texto) y Tecnica 2 (booleana con col()):**

In [13]:
df_mediciones.filter("fuera_de_rango = 1 AND canal = 'sensor_iot'").show(5)

+-----------+-----------+----------+-----+----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+-------------+--------------+--------------------+---------------+
|medicion_id|estacion_id|     fecha| hora|     canal|  ph|temperatura_c|turbidez_ntu|oxigeno_disuelto_mgl|conductividad_us_cm|solidos_disueltos_ppm|nitratos_mgl|fosfatos_mgl|coliformes_fecales_cfu|observaciones|fuera_de_rango|      fuente_dataset|fecha_procesado|
+-----------+-----------+----------+-----+----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+-------------+--------------+--------------------+---------------+
|          2|   EST-0063|2024-11-12|10:35|sensor_iot|6.29|          8.2|       10.34|                 7.0|              431.8|                243.2|        NULL|        NULL|                  NULL|           

In [14]:
df_mediciones.filter((col("fuera_de_rango") == 1) & (col("oxigeno_disuelto_mgl").between(2, 5))).show(5)
print("Con canal nulo (eqNullSafe):", df_mediciones.filter(col("canal").eqNullSafe(None)).count())

+-----------+-----------+----------+-----+-----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+--------------------+--------------+--------------------+---------------+
|medicion_id|estacion_id|     fecha| hora|      canal|  ph|temperatura_c|turbidez_ntu|oxigeno_disuelto_mgl|conductividad_us_cm|solidos_disueltos_ppm|nitratos_mgl|fosfatos_mgl|coliformes_fecales_cfu|       observaciones|fuera_de_rango|      fuente_dataset|fecha_procesado|
+-----------+-----------+----------+-----+-----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+--------------------+--------------+--------------------+---------------+
|          9|   EST-0153|2024-05-12|19:22|laboratorio|5.96|          7.1|        1.96|                3.59|              283.2|                153.4|        2.74|       0.006|         

**Orden — Tecnica 1 (orderBy()) y Tecnica 2 (sort()):**

In [15]:
df_mediciones.orderBy(col("estacion_id").asc(), col("oxigeno_disuelto_mgl").asc()).select(
    "estacion_id", "oxigeno_disuelto_mgl", "fuera_de_rango"
).show(5)

+-----------+--------------------+--------------+
|estacion_id|oxigeno_disuelto_mgl|fuera_de_rango|
+-----------+--------------------+--------------+
|   EST-0001|                1.47|             1|
|   EST-0001|                1.55|             1|
|   EST-0001|                1.59|             1|
|   EST-0001|                1.62|             1|
|   EST-0001|                1.93|             1|
+-----------+--------------------+--------------+
only showing top 5 rows


In [16]:
df_mediciones.sort(col("fuera_de_rango").desc(), col("ph").asc_nulls_last()).select(
    "estacion_id", "ph", "fuera_de_rango"
).show(5)

+-----------+----+--------------+
|estacion_id|  ph|fuera_de_rango|
+-----------+----+--------------+
|   EST-0176| 4.5|             1|
|   EST-0060| 4.5|             1|
|   EST-0140| 4.5|             1|
|   EST-0006| 4.5|             1|
|   EST-0054|4.52|             1|
+-----------+----+--------------+
only showing top 5 rows


**Duplicados — diagnostico y tratamiento con 2 tecnicas (incluida Window+row_number):**

In [17]:
dup = df_mediciones.groupBy("estacion_id", "fecha", "hora").count().filter("count > 1")
print("Grupos duplicados (estacion_id+fecha+hora):", dup.count())

[Stage 42:================================================>         (5 + 1) / 6]

Grupos duplicados (estacion_id+fecha+hora): 4175


In [18]:
total = df_mediciones.count()
sin_dup_fila_completa = df_mediciones.distinct().count()
print(f"Total: {total}, sin duplicar (fila completa): {sin_dup_fila_completa}")

[Stage 51:================================================>         (5 + 1) / 6]

Total: 1250000, sin duplicar (fila completa): 1250000


In [19]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

ventana_dedup = Window.partitionBy("estacion_id", "fecha", "hora").orderBy(col("medicion_id").desc())
df_mediciones_unico = (
    df_mediciones
    .withColumn("row_num", row_number().over(ventana_dedup))
    .filter(col("row_num") == 1)
    .drop("row_num")
)
print(f"Mediciones: {df_mediciones.count():,} -> tras deduplicar: {df_mediciones_unico.count():,}")

[Stage 60:================================================>         (5 + 1) / 6]

Mediciones: 1,250,000 -> tras deduplicar: 1,245,817


**Nulos: tratamiento con criterio documentado:**

In [20]:
# nitratos/fosfatos/coliformes solo laboratorio los mide.
df_valido = df_mediciones_unico.na.drop(subset=["medicion_id", "estacion_id", "fecha", "hora"])
print(f"Filas antes: {df_mediciones_unico.count():,}, despues de na.drop(subset criticos): {df_valido.count():,}")

[Stage 72:================================================>         (5 + 1) / 6]

Filas antes: 1,245,817, despues de na.drop(subset criticos): 1,245,817


In [21]:
df_estaciones_limpio = df_estaciones.dropDuplicates(["estacion_id"])
df_gold_base = df_valido.join(df_estaciones_limpio, on="estacion_id", how="left")
print("Columnas Gold:", df_gold_base.columns)
print("Filas:", df_gold_base.count())
df_gold_base = df_gold_base.cache()

Columnas Gold: ['estacion_id', 'medicion_id', 'fecha', 'hora', 'canal', 'ph', 'temperatura_c', 'turbidez_ntu', 'oxigeno_disuelto_mgl', 'conductividad_us_cm', 'solidos_disueltos_ppm', 'nitratos_mgl', 'fosfatos_mgl', 'coliformes_fecales_cfu', 'observaciones', 'fuera_de_rango', 'fuente_dataset', 'fecha_procesado', 'nombre_estacion', 'region', 'cuenca', 'tipo_fuente', 'latitud', 'longitud', 'fecha_instalacion']


[Stage 78:================================================>         (5 + 1) / 6]

Filas: 1245817


In [22]:
ruta_gold = f"{ARTIFACTS}/mediciones_particionado"

(
    df_gold_base
    .repartition(4)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("region")
    .save(ruta_gold)
)

import os
print("Particiones (region):", sorted(os.listdir(ruta_gold)))

Particiones (region): ['._SUCCESS.crc', '_SUCCESS', 'region=Arequipa', 'region=Cusco', 'region=Junín', 'region=Lambayeque', 'region=Lima', 'region=Loreto', 'region=Piura', 'region=Puno']


In [23]:
df_verificacion = spark.read.parquet(ruta_gold)
assert df_verificacion.count() == df_gold_base.count()
print(f"Verificado: {df_verificacion.count():,} filas, ida y vuelta sin perdida.")

Verificado: 1,245,817 filas, ida y vuelta sin perdida.


In [24]:
df_verificacion.filter(col("region") == "Puno").explain(True)

== Parsed Logical Plan ==
'Filter '`=`('region, Puno)
+- Relation [estacion_id#1499,medicion_id#1500L,fecha#1501,hora#1502,canal#1503,ph#1504,temperatura_c#1505,turbidez_ntu#1506,oxigeno_disuelto_mgl#1507,conductividad_us_cm#1508,solidos_disueltos_ppm#1509,nitratos_mgl#1510,fosfatos_mgl#1511,coliformes_fecales_cfu#1512,observaciones#1513,fuera_de_rango#1514,fuente_dataset#1515,fecha_procesado#1516,nombre_estacion#1517,cuenca#1518,tipo_fuente#1519,latitud#1520,longitud#1521,fecha_instalacion#1522,region#1523] parquet

== Analyzed Logical Plan ==
estacion_id: string, medicion_id: bigint, fecha: string, hora: string, canal: string, ph: double, temperatura_c: double, turbidez_ntu: double, oxigeno_disuelto_mgl: double, conductividad_us_cm: double, solidos_disueltos_ppm: double, nitratos_mgl: double, fosfatos_mgl: double, coliformes_fecales_cfu: double, observaciones: string, fuera_de_rango: int, fuente_dataset: string, fecha_procesado: date, nombre_estacion: string, cuenca: string, tipo_fue

**Capas Bronze/Silver/Gold:**
- **Bronze:** `estaciones_agua.csv` / `mediciones_calidad_agua.parquet` tal como llegan.
- **Silver:** `df_valido` — esquema validado, duplicados resueltos, nulos criticos tratados.
- **Gold:** `mediciones_particionado/` — integrado con estaciones, particionado por
  `region`, listo para el componente ML (Bloque 4).

## 4. Componente ML distribuido (S4) — Clasificacion

**Objetivo (Brief):** estimar la probabilidad de que una estacion tenga una
frecuencia alta de mediciones fuera de rango el **mes siguiente**, a partir de su
historico agregado mensual (Gold del Bloque 3). A diferencia de la dimension de
Jhenderson (regresion), aqui la variable objetivo es categorica (alerta si/no) —
la guia S4 (2.1) permite usar `LogisticRegression`/`RandomForestClassifier` en vez
de `LinearRegression`/`RandomForestRegressor`, con la misma disciplina de comparar
antes de guardar.

In [25]:
from pyspark.sql.functions import to_date, date_format, dense_rank, lead, sum as spark_sum

df_mes = df_verificacion.withColumn("AnioMes", date_format(to_date(col("fecha")), "yyyy-MM"))

df_agregado = (
    df_mes.groupBy("estacion_id", "AnioMes")
    .agg(
        spark_count("*").alias("total_mediciones"),
        spark_sum("fuera_de_rango").alias("mediciones_fuera_rango"),
        avg("ph").alias("ph_promedio"),
        avg("turbidez_ntu").alias("turbidez_promedio"),
        avg("temperatura_c").alias("temperatura_promedio"),
        avg("oxigeno_disuelto_mgl").alias("oxigeno_promedio"),
        avg("conductividad_us_cm").alias("conductividad_promedio"),
        avg("solidos_disueltos_ppm").alias("solidos_promedio"),
    )
    .withColumn("pct_fuera_rango", col("mediciones_fuera_rango") / col("total_mediciones") * 100)
)
print("Filas agregadas (estacion x mes):", df_agregado.count())
df_agregado.orderBy(col("pct_fuera_rango").desc()).show(5)

Filas agregadas (estacion x mes): 4320


+-----------+-------+----------------+----------------------+------------------+-----------------+--------------------+-----------------+----------------------+------------------+------------------+
|estacion_id|AnioMes|total_mediciones|mediciones_fuera_rango|       ph_promedio|turbidez_promedio|temperatura_promedio| oxigeno_promedio|conductividad_promedio|  solidos_promedio|   pct_fuera_rango|
+-----------+-------+----------------+----------------------+------------------+-----------------+--------------------+-----------------+----------------------+------------------+------------------+
|   EST-0136|2024-06|             254|                    81| 7.053307086614174|5.672834645669291|  15.863385826771658|6.864173228346455|     375.2496062992126|240.26811023622048| 31.88976377952756|
|   EST-0128|2024-12|             283|                    90| 7.195265017667844| 5.86487632508834|  15.515547703180207|6.724134275618375|    379.32826855123676|240.79257950530035|31.802120141342755|
|   E

In [26]:
ventana_mes = Window.orderBy("AnioMes")
mapa_meses = df_agregado.select("AnioMes").distinct().withColumn("mes_numero", dense_rank().over(ventana_mes))
df_agregado = df_agregado.join(mapa_meses, on="AnioMes", how="left")

mediana_pct = df_agregado.approxQuantile("pct_fuera_rango", [0.5], 0.01)[0]
print("Mediana pct_fuera_rango (umbral de alerta):", mediana_pct)

ventana_lead = Window.partitionBy("estacion_id").orderBy("mes_numero")
df_con_objetivo = (
    df_agregado
    .withColumn("pct_fuera_rango_siguiente", lead("pct_fuera_rango", 1).over(ventana_lead))
)
df_con_objetivo = df_con_objetivo.withColumn(
    "alerta_siguiente_mes",
    when(col("pct_fuera_rango_siguiente") > mediana_pct, 1).otherwise(0)
)

df_dataset = df_con_objetivo.na.drop(subset=["pct_fuera_rango_siguiente"]).cache()
print("Filas utilizables (con mes siguiente conocido):", df_dataset.count())
df_dataset.groupBy("alerta_siguiente_mes").count().show()

26/09/11 04:31:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 04:31:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Mediana pct_fuera_rango (umbral de alerta): 22.45614035087719


26/09/11 04:31:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 04:31:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 04:31:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 04:31:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 04:31:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 04:31:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
          

Filas utilizables (con mes siguiente conocido): 4140
+--------------------+-----+
|alerta_siguiente_mes|count|
+--------------------+-----+
|                   1| 2117|
|                   0| 2023|
+--------------------+-----+



In [27]:
PREDICTORES = ["mes_numero", "pct_fuera_rango", "ph_promedio", "turbidez_promedio",
               "temperatura_promedio", "oxigeno_promedio", "conductividad_promedio", "solidos_promedio"]
OBJETIVO = "alerta_siguiente_mes"

for c in PREDICTORES:
    print(f"{c:24s} correlacion con {OBJETIVO}: {df_dataset.stat.corr(c, OBJETIVO):.4f}")

mes_numero               correlacion con alerta_siguiente_mes: -0.0087
pct_fuera_rango          correlacion con alerta_siguiente_mes: -0.0006
ph_promedio              correlacion con alerta_siguiente_mes: -0.0095
turbidez_promedio        correlacion con alerta_siguiente_mes: -0.0298
temperatura_promedio     correlacion con alerta_siguiente_mes: -0.0032
oxigeno_promedio         correlacion con alerta_siguiente_mes: 0.0053
conductividad_promedio   correlacion con alerta_siguiente_mes: 0.0244
solidos_promedio         correlacion con alerta_siguiente_mes: 0.0291


Como el objetivo es un estado FUTURO, el split es cronologico, no
aleatorio, misma razon que en la dimension de Jhenderson (S4, Tabla 3): un split
aleatorio filtraria informacion del futuro hacia el entrenamiento.

In [28]:
max_mes = df_dataset.agg({"mes_numero": "max"}).collect()[0][0]
corte = max_mes - 4
print("mes_numero maximo:", max_mes, " corte de train/test:", corte)

df_train = df_dataset.filter(col("mes_numero") <= corte)
df_test = df_dataset.filter(col("mes_numero") > corte)
print("Train:", df_train.count(), "Test:", df_test.count())

mes_numero maximo: 23  corte de train/test: 19
Train: 3420 Test: 720


In [29]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

ensamblador = VectorAssembler(inputCols=PREDICTORES, outputCol="features")
df_train_ml = ensamblador.transform(df_train).select("features", OBJETIVO)
df_test_ml = ensamblador.transform(df_test).select("features", OBJETIVO)

def evaluar(pred, nombre):
    auc = BinaryClassificationEvaluator(labelCol=OBJETIVO, metricName="areaUnderROC").evaluate(pred)
    f1 = MulticlassClassificationEvaluator(labelCol=OBJETIVO, metricName="f1").evaluate(pred)
    acc = MulticlassClassificationEvaluator(labelCol=OBJETIVO, metricName="accuracy").evaluate(pred)
    print(f"{nombre}: AUC={auc:.4f}  F1={f1:.4f}  Accuracy={acc:.4f}")
    return {"AUC": auc, "F1": f1, "Accuracy": acc}

lr_base = LogisticRegression(featuresCol="features", labelCol=OBJETIVO)
modelo_base = lr_base.fit(df_train_ml)
print("Coeficientes:", modelo_base.coefficients)
print("Intercepto:", modelo_base.intercept)
resultados_base = evaluar(modelo_base.transform(df_test_ml), "LogisticRegression base")

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


Coeficientes: [-0.0006308591754770102,-0.007886824587662416,-0.6061129100589133,-0.21367029916244115,0.042892688748647295,-0.10499261992121955,-0.011330355235493932,0.02510915402689089]
Intercepto: 4.12863344527658
LogisticRegression base: AUC=0.5092  F1=0.4939  Accuracy=0.5153


In [30]:
configuraciones = [
    {"nombre": "Sin regularizacion", "regParam": 0.0, "elasticNetParam": 0.0},
    {"nombre": "Ridge (L2)", "regParam": 0.1, "elasticNetParam": 0.0},
    {"nombre": "Elastic Net (L1+L2)", "regParam": 0.1, "elasticNetParam": 0.5},
]
comparacion_configs = []
for config in configuraciones:
    lr = LogisticRegression(featuresCol="features", labelCol=OBJETIVO,
                             regParam=config["regParam"], elasticNetParam=config["elasticNetParam"])
    modelo = lr.fit(df_train_ml)
    r = evaluar(modelo.transform(df_test_ml), config["nombre"])
    r["Configuracion"] = config["nombre"]
    comparacion_configs.append(r)

import pandas as pd
pd.DataFrame(comparacion_configs)[["Configuracion","AUC","F1","Accuracy"]]

Sin regularizacion: AUC=0.5092  F1=0.4939  Accuracy=0.5153
Ridge (L2): AUC=0.5075  F1=0.4489  Accuracy=0.4931
Elastic Net (L1+L2): AUC=0.5000  F1=0.3287  Accuracy=0.4958


,Configuracion,AUC,F1,Accuracy
0,Sin regularizacion,0.509248,0.493940,0.515278
1,Ridge (L2),0.507458,0.448929,0.493056
2,Elastic Net (L1+L2),0.500000,0.328714,0.495833


In [31]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol="features", labelCol=OBJETIVO, numTrees=50, maxDepth=6, seed=42)
modelo_rf = rf.fit(df_train_ml)
resultados_rf = evaluar(modelo_rf.transform(df_test_ml), "Random Forest")

importancias = sorted(zip(PREDICTORES, modelo_rf.featureImportances.toArray()), key=lambda x: -x[1])
for v, imp in importancias:
    print(f"{v:24s} {imp:.4f}")

Random Forest: AUC=0.5127  F1=0.4800  Accuracy=0.4972
turbidez_promedio        0.1499
temperatura_promedio     0.1420
oxigeno_promedio         0.1418
pct_fuera_rango          0.1284
solidos_promedio         0.1153
conductividad_promedio   0.1144
ph_promedio              0.1070
mes_numero               0.1013


In [32]:
tabla_final = pd.DataFrame(comparacion_configs + [{"Configuracion":"Random Forest", **resultados_rf}])[["Configuracion","AUC","F1","Accuracy"]]
tabla_final

,Configuracion,AUC,F1,Accuracy
0,Sin regularizacion,0.509248,0.493940,0.515278
1,Ridge (L2),0.507458,0.448929,0.493056
2,Elastic Net (L1+L2),0.500000,0.328714,0.495833
3,Random Forest,0.512705,0.480044,0.497222


**Evaluation (conclusion de negocio):** los cuatro modelos quedan muy cerca de
AUC=0.50 (equivalente a una moneda al aire) y Accuracy≈0.50 — ninguno logra separar
de forma confiable las estaciones que tendran alta frecuencia de mediciones fuera de
rango el mes siguiente de las que no. Las correlaciones de la celda anterior ya
anticipaban esto: ninguna variable agregada (pH, turbidez, temperatura, oxigeno,
conductividad, solidos) predice el estado de alerta del mes siguiente mejor que el
azar. **Validacion de negocio:** con el historico disponible (24 meses) y estas
variables, el modelo NO deberia usarse todavia para priorizar auditorias de forma
automatica — se necesitaria mas historia o variables adicionales (ej. eventos de
lluvia, actividad industrial cercana) antes de confiar en esta prediccion para una
decision real, exactamente el tipo de conclusion honesta que el caso Zillow (S4, 1.6)
recomienda sacar antes de desplegar un modelo.

In [33]:
modelo_ganador = modelo_base  # LogisticRegression sin regularizacion: mejor AUC de los cuatro
modelo_ganador.write().overwrite().save(f"{ARTIFACTS}/modelo_alerta_siguiente_mes")
print("Modelo guardado en", f"{ARTIFACTS}/modelo_alerta_siguiente_mes")

Modelo guardado en ./artifacts/modelo_alerta_siguiente_mes


## Error o hallazgo

**Que ocurrio:** al definir la variable objetivo, la primera version usaba
`pct_fuera_rango` (continua) directamente como si fuera categorica -- el
`BinaryClassificationEvaluator` fallo porque esperaba una etiqueta 0/1, no un
porcentaje.

**Como lo identifique:** el traceback senalo un error de tipo en la columna label,
y al revisar `df_dataset.printSchema()` confirme que `pct_fuera_rango_siguiente`
era `double`, no una clase binaria.

**Como lo resolvi:** binaricé el objetivo con un umbral (la mediana historica de
`pct_fuera_rango`, ~22.5%): si el porcentaje del mes siguiente supera esa mediana,
`alerta_siguiente_mes = 1`. Documentar el umbral explicitamente, en vez de un
punto de corte arbitrario, es lo que hace que la definicion de "alerta" sea
reproducible y defendible.

## Reflexion tecnica breve

La disciplina de S4 (entrenar, evaluar con varias metricas, comparar contra al
menos una configuracion alternativa, guardar el ganador real) se mantiene identica
al cambiar de regresion a clasificacion, solo cambian los evaluadores
(`BinaryClassificationEvaluator`/`MulticlassClassificationEvaluator` en vez de
`RegressionEvaluator`) y el algoritmo (`LogisticRegression`/`RandomForestClassifier`
en vez de `LinearRegression`/`RandomForestRegressor`). Que ningun modelo supere el
azar (AUC≈0.50) no invalida el pipeline: confirma, con evidencia y no por intuicion,
que las variables fisico-quimicas disponibles no bastan para anticipar el estado de
alerta de una estacion, un hallazgo tan valido para la toma de decision como un
modelo que si funcionara.